In [ ]:
#Algorithm outline and testing

In [ ]:
# === PART 1: DETECT & LOCALIZE FLIES PER FRAME ===
""" goal is to record latency of frame substraction, ensure that program can keep track of multiple flies in each frame """

#imports
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

In [ ]:
#constants
video_path = ".avi" #path to AVI file
DOWNSCALE = 1.0 #<1.0 to downscale frames for speed
CROP_REGION = 2.0 # crop to "bait part" of the video
THRESH_VAL = 25 # frame subtraction threshold
MIN_AREA = 5 # minimum blob area (pixels)
MAX_AREA = 500 # max blob area (pixels)
MAX_TRACK_DIST = 30 # max distance for track association (pixels)
HISTORY = 50 # frame to keep for plotting

In [ ]:
#tracking class
class Track: 
    def __init__(self, track_id, centroid): 
        self.id = track_id
        self.centroids = deque(maxlen=HISTORY)
        self.centroids.append(centroid)
        self.last_seen = 0

    @property
    def last_position(self): 
        return self.centroids[-1] # returns last position stored in centroid

In [ ]:
# utility functions
def preprocess(frame): 
    #TODO figure out what region to crop appropriately as well, should we have a way to determine "approx" where this will be every time
    if DOWNSCALE != 1.0: 
        frame = cv2.resize(frame, None, fx=DOWNSCALE, fy=DOWNSCALE)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
return gray # do i need this if its already in grayscale video? 

def detect_moving_objects(prev_gray, curr_gray): 
    #TODO test how well this works for cases where flies aren't moving much around the bait
    diff = cv2.absdiff(curr_gray, prev_gray)
    _, thres = cv2.threshold(diff, THRESH_VAL, 255, cv2.THRESH_BINARY)
    thresh = cv2.medianBlur(thresh, 5)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = [] 
    for c in contours: 
        area = cv2.contourArea(c)
        if MIN_AREA <= area <= MAX_AREA: # looking for objects within specifed range size TODO figure out how to determine this 
            M = cv2.moments(c)
            if M['m00'] != 0: 
                cx = int(M['m10']/ M['m00'])
                cy = int(M['m01']/ M['m00'])
                detections.append((cx, cy))
    return detections, thresh

def associate_detections_to_tracking(detections, tracks, next_id): 
    assigned = set() 
    for track in tracks: 
        min_dist = float('inf')
        best_det = None
        for d in detections: 
            if d in assigned: 
                continue
            dist = np.linalg.norm(np.array(track.last_position) - np.array(d))
            if dist < min_dist and dist < MAX_TRACK_DIST: 
                min_dist = dist
                best_det = d
        if best_det is not None: 
            track.centroids.append(best_det)
            track.last_seen = 0
            assigned.add(best_det)
        else: 
            track.last_seen += 1

    for d in detections: 
        if d not in assigned: 
            tracks.append(Track(next_id, d))
            next_id += 1

    tracks = [t for t in tracks if t.last_seen < 5] #TODO may need to adjust < 5 starting threshold
    return tracks, next_id
    

In [ ]:
# MAIN PROCESSING LOOP